# **04. Funnel Conversion & Performance Analysis**

#### **Project Overview**

The funnel presented in this analysis is an __analytical conversion funnel__ derived from the Bank Marketing dataset, which contains data from direct telemarketing campaigns conducted to sell __term deposit subscriptions__. The goal is to evaluate the reduction from the overall customer base to successful subscriptions, identify high-performing customer segments, and understand factors that influence the bank's overall conversion rate.
The intermediate stages represent important campaign characteristics and customer attributes rather than sequential steps in an individual customer's journey.


## **1. Load Clean Dataset**

In [1]:
## Import commonly used libraries
import pandas as pd 
import numpy as np  
import sidetable
import sklearn
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from glob import glob
from itertools import combinations


In [2]:

## Dispaly Settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option("display.float_format", "{:.2f}".format)

import warnings
warnings.filterwarnings('ignore')

from IPython.display import display
from IPython.display import Markdown

# Display markdown formatted output like bold, italic bold etc.'''
def display_md(string):
    display(Markdown(string))

  

In [3]:
  
# Read the data
files = []
parent_path = Path.cwd().parent
data_path = parent_path.joinpath('data', 'processed')
for file in data_path.glob('*'):
    files.append(file)
    print(file.name)
    print(files.index(file), " ", file)
    

.gitkeep
0   e:\Bank-Telemarketing\data\processed\.gitkeep
clean_data.csv
1   e:\Bank-Telemarketing\data\processed\clean_data.csv
funnel_data.csv
2   e:\Bank-Telemarketing\data\processed\funnel_data.csv


In [4]:
# Loading clean file
display_md("**Loading Clean-Funnel Dataset....**")
try:
    data = pd.read_csv(files[2])
    display(data.head(2))
    display_md(f"**Dataset Shape :** {data.shape}")
except Exception as e:
    print(f"Data Loading Error : {e}")


**Loading Clean-Funnel Dataset....**

,age,age_basket,job,marital,education,balance,balance_basket,default,housing,loan,contact,day,month,duration,duration_mins,duration_bucket,campaign,campaign_capped,pdays,previous,poutcome,term_deposit,term_deposit_numeric
0,58,50-66,management,married,tertiary,2143,High (2K+),no,yes,no,unknown,5,may,261,4.35,3-5 mins,1,1,-1,0,unknown,no,0
1,44,35-49,technician,single,secondary,29,Low (0-500),no,yes,no,unknown,5,may,151,2.52,1-3 mins,1,1,-1,0,unknown,no,0


**Dataset Shape :** (45211, 23)

## **2. Analytical Funnel Stage**

In [40]:
display_md("**Analytical Funnel Stage :**")

import pandas as pd

total = len(data)

funnel_df = pd.DataFrame({
    "Analytical Funnel Stage": [
        "Stage 1 — Total Customers Contacted",
        "Stage 2 — Contacted via Cellular",
        "Stage 3 — Previously Contacted",
        "Stage 4 — Subscribed"
    ],
    "Customers": [
        total,
        len(data[data['contact'] == 'cellular']),
        len(data[data['previous'] > 0]),
        len(data[data['term_deposit'] == 'yes'])
    ],
    "% of Total": [
        round(total / total * 100, 1),
        round(len(data[data['contact'] == 'cellular']) / total * 100, 1),
        round(len(data[data['previous'] > 0]) / total * 100, 1),
        round(len(data[data['term_deposit'] == 'yes']) / total * 100, 1)
    ]
})

# Compute conversion from previous stage
funnel_df['Conversion from Previous Stage(%)'] = np.round(
    100 * funnel_df['Customers'] / funnel_df['Customers'].shift(1),
    1
)

# # First stage has no previous stage
# funnel_df.loc[0, 'Conversion from Previous Stage(%)'] = 0

# Compute drop-off from previous stage
funnel_df["Drop-off from Previous Stage(%)"] = np.round(
    100 - funnel_df["Conversion from Previous Stage(%)"],
    1
)

display(funnel_df.style.hide(axis='index'))


**Analytical Funnel Stage :**

Analytical Funnel Stage,Customers,% of Total,Conversion from Previous Stage(%),Drop-off from Previous Stage(%)
Stage 1 — Total Customers Contacted,45211,100.000000,nan,nan
Stage 2 — Contacted via Cellular,29285,64.800000,64.800000,35.200000
Stage 3 — Previously Contacted,8257,18.300000,28.200000,71.800000
Stage 4 — Subscribed,5289,11.700000,64.100000,35.900000


#### **2.1. Drop-off Insights**

**Drop-off 1 — Stage 1 to Stage 2**

In [ ]:
# Total customers
total = len(data)

# Stage 1 → Stage 2
cellular = data[data['contact'] == 'cellular']
unknown = data[data['contact'] == 'unknown']

# Counts
cellular_count = len(cellular)
unknown_count = len(unknown)

# Drop-off
dropoff_count = total - cellular_count
dropoff_pct = dropoff_count / total * 100

# Conversion rates
cellular_conv = (
    len(cellular[cellular['term_deposit'] == 'yes']) /
    cellular_count * 100
)

unknown_conv = (
    len(unknown[unknown['term_deposit'] == 'yes']) /
    unknown_count * 100
)

# Improvement factor
improvement = cellular_conv / unknown_conv

# Estimated additional subscriptions if unknown contacts
# had converted at the cellular conversion rate
expected_unknown_yes = unknown_count * (cellular_conv / 100)
actual_unknown_yes = len(unknown[unknown['term_deposit'] == 'yes'])

extra_subscriptions = expected_unknown_yes - actual_unknown_yes

# Results
print(f"Stage 1 Customers          : {total:,}")
print(f"Stage 2 Cellular           : {cellular_count:,}")

print(f"\nDrop-off: {dropoff_count:,} customers ({dropoff_pct:.1f}%)")

print(f"\nUnknown contacts          : {unknown_count:,}")

print(f"Cellular Conversion Rate  : {cellular_conv:.1f}%")
print(f"Unknown Conversion Rate   : {unknown_conv:.1f}%")

display_md(f"**Cellular performs {improvement:.1f}x better than unknown.**")

print(f"Estimated additional subscriptions if all unknown contacts")
print(f"performed like cellular : {extra_subscriptions:.0f}")


Stage 1 Customers          : 45,211
Stage 2 Cellular           : 29,285

Drop-off: 15,926 customers (35.2%)

Unknown contacts          : 13,020
Cellular Conversion Rate  : 14.9%
Unknown Conversion Rate   : 4.1%


**Cellular performs 3.7x better than unknown.**

Estimated additional subscriptions if all unknown contacts
performed like cellular : 1412


**Drop-off 2 — Stage 2 to Stage 3**

In [49]:
# Total customers
total = len(data)

# Customers with previous contact
previous_contact = data[data["previous"] > 0]

# Customers with no previous contact
new_customers = data[data["previous"] == 0]

# Customers with a previous successful campaign
previous_success = data[data["poutcome"] == "success"]

# Stage 2 → Stage 3 drop-off
dropoff = len(data[data["contact"] == "cellular"]) - len(previous_contact)

# Percentage of customers with no prior relationship
new_pct = len(new_customers) / total * 100

# Conversion rate: new customers
new_conversion = (
    len(new_customers[new_customers["term_deposit"] == "yes"])
    / len(new_customers)
    * 100
)

# Conversion rate: previously successful customers
success_conversion = (
    len(previous_success[previous_success["term_deposit"] == "yes"])
    / len(previous_success)
    * 100
)

# Improvement factor
improvement = success_conversion / new_conversion

print(f"Customers Reduction (Stage 2 → Stage 3): {dropoff:,}")
print(f"Customers with no prior relationship: {len(new_customers):,} ({new_pct:.1f}%)")
print(f"New customer conversion: {new_conversion:.1f}%")
print(f"Previously successful customer conversion: {success_conversion:.1f}%")
display_md(f"**Previously successful customers convert {improvement:.1f}x better.**")


Customers Reduction (Stage 2 → Stage 3): 21,028
Customers with no prior relationship: 36,954 (81.7%)
New customer conversion: 9.2%
Previously successful customer conversion: 64.7%


**Previously successful customers convert 7.1x better.**

**Drop-off 3 — Stage 3 to Stage 4**

In [51]:
# Single call customers
single_call = data[data["campaign"] == 1]

# Customers contacted 6+ times
six_plus_calls = data[data["campaign"] >= 6]

# Conversion rates
single_call_conversion = (
    len(single_call[single_call["term_deposit"] == "yes"]) /
    len(single_call) * 100
)

six_plus_conversion = (
    len(six_plus_calls[six_plus_calls["term_deposit"] == "yes"]) /
    len(six_plus_calls) * 100
)

# Conversion decline
conversion_drop = single_call_conversion - six_plus_conversion

# Results
print(f"Single call customers: {len(single_call):,}")
print(f"6+ call customers: {len(six_plus_calls):,}")

print(f"\nSingle call conversion rate: {single_call_conversion:.1f}%")
print(f"6+ calls conversion rate: {six_plus_conversion:.1f}%")

display_md(f"**Conversion decline after 6+ calls: {conversion_drop:.1f} percentage points**")


Single call customers: 17,544
6+ call customers: 4,355

Single call conversion rate: 14.6%
6+ calls conversion rate: 5.8%


**Conversion decline after 6+ calls: 8.8 percentage points**

## **3. Baseline ROI vs Improved ROI**

In [53]:
# Business assumptions
revenue_per_subscription = 1000   # revenue/value per successful subscription
cost_per_contact = 5             # cost per customer contact

# Baseline metrics
total_customers = len(data)
baseline_subscriptions = len(data[data["term_deposit"] == "yes"])

baseline_revenue = baseline_subscriptions * revenue_per_subscription
baseline_cost = total_customers * cost_per_contact

baseline_roi = (
    (baseline_revenue - baseline_cost) /
    baseline_cost * 100
)


# Improved scenario assumptions

# Example improvements:
# 1. Unknown contacts moved to cellular (+1400 subscriptions)
# 2. Better targeting reduces ineffective contacts

additional_subscriptions = 1400

improved_subscriptions = baseline_subscriptions + additional_subscriptions

improved_revenue = improved_subscriptions * revenue_per_subscription

# Assume same campaign size/cost
improved_cost = baseline_cost

improved_roi = (
    (improved_revenue - improved_cost) /
    improved_cost * 100
)


# ROI improvement
roi_gain = improved_roi - baseline_roi


print(f"Baseline subscriptions: {baseline_subscriptions:,}")
print(f"Improved subscriptions: {improved_subscriptions:,}")

print(f"\nBaseline ROI: {baseline_roi:.2f}%")
print(f"Improved ROI: {improved_roi:.2f}%")

display_md(f"**ROI improvement: +{roi_gain:.2f} percentage points**")


Baseline subscriptions: 5,289
Improved subscriptions: 6,689

Baseline ROI: 2239.70%
Improved ROI: 2859.01%


**ROI improvement: +619.32 percentage points**

- The ROI improvement shows the financial benefit of optimizing targeting and contact strategy.

| Metric | Baseline Campaign | Improved Campaign |
|--------|------------------:|------------------:|
| Total Customers Contacted | 45,211 | 45,211 |
| Successful Subscriptions | 5,289 | 6,689 |
| Conversion Rate | 11.7% | 14.8% |
| Revenue per Subscription | $1,000 | $1,000 |
| Total Revenue | $5,289,000 | $6,689,000 |
| Campaign Cost (@ $5/contact) | $226,055 | $226,055 |
| Net Profit | $5,062,945 | $6,462,945 |
| ROI | 2,239.7% | 2,859.0% |
| ROI Improvement | — | +619.3 percentage points |